# Hands-on Exercise 4 — Register & Promote a Model
### AI Operations (AIOps) — MLflow Deep Dive | ~10–15 minutes

**Referenced in:** *MLflow Deep Dive Slide Deck*, Section 4 (The Model Registry)

**Objective:** register a model under a stable name, create a second version, move versions through
the Registry lifecycle using both the classic **Stages** API and the newer **Aliases** API, and load
the model back by stage and by alias.

**Steps (from the slide deck):**
1. Register the model from your Exercise 3 run under the name `"my-classifier"`.
2. Register a second version by re-running training with different hyperparameters and registering again.
3. Transition version 1 to `"Staging"` and version 2 to `"Production"` (with `archive_existing_versions=True`).
4. Set an alias `"@champion"` pointing at whichever version has the best accuracy.
5. Write a short script that loads the model via `models:/my-classifier/Production` and via the
   `@champion` alias, and confirm both work.

**Deliverable:** a registered model with ≥ 2 versions, a visible stage/alias history in the UI, and a
script proving stage- and alias-based loading both work.

> **Prerequisite:** the MLflow Tracking Server must still be running at `http://localhost:5000`,
> and you should have completed Exercise 3 (or run the cell below, which retrains a model if needed).

## Step 0 — Setup

In [ ]:
# !pip install mlflow scikit-learn pandas --quiet
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("iris-classifier")
client = MlflowClient()

MODEL_NAME = "my-classifier"

X, y = load_iris(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Step 1 — Train + log two candidate runs (if you don't already have them from Exercise 3)

In [ ]:
def train_and_log(n_estimators, max_depth, run_name):
    with mlflow.start_run(run_name=run_name) as run:
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        acc = accuracy_score(y_test, preds)
        signature = infer_signature(X_train.iloc[:5], model.predict(X_train.iloc[:5]))

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_metric("accuracy", acc)
        mlflow.sklearn.log_model(model, name="model", signature=signature,
                                  input_example=X_train.iloc[:5])
        return run.info.run_id, acc

run_id_a, acc_a = train_and_log(100, 5, "candidate-A")
run_id_b, acc_b = train_and_log(250, 8, "candidate-B")
print(f"candidate-A: run_id={run_id_a}  accuracy={acc_a:.4f}")
print(f"candidate-B: run_id={run_id_b}  accuracy={acc_b:.4f}")

## Step 1 (continued) — Register both as versions of `my-classifier`

In [ ]:
result_a = mlflow.register_model(model_uri=f"runs:/{run_id_a}/model", name=MODEL_NAME)
result_b = mlflow.register_model(model_uri=f"runs:/{run_id_b}/model", name=MODEL_NAME)

print(f"Registered {result_a.name} version {result_a.version}  (from candidate-A)")
print(f"Registered {result_b.name} version {result_b.version}  (from candidate-B)")

## Step 3 — Transition versions with the classic Stages API

In [ ]:
client.transition_model_version_stage(
    name=MODEL_NAME, version=result_a.version, stage="Staging",
)
print(f"Version {result_a.version} -> Staging")

client.transition_model_version_stage(
    name=MODEL_NAME, version=result_b.version, stage="Production",
    archive_existing_versions=True,
)
print(f"Version {result_b.version} -> Production (older Production versions auto-archived)")

## Step 4 — Set a `@champion` alias on the best-accuracy version
The Aliases API is the modern recommended alternative/complement to Stages.

In [ ]:
best_version = result_a.version if acc_a >= acc_b else result_b.version
best_acc = max(acc_a, acc_b)

client.set_registered_model_alias(name=MODEL_NAME, alias="champion", version=best_version)
print(f"@champion -> version {best_version}  (accuracy={best_acc:.4f})")

# Also tag it for extra audit metadata
client.set_model_version_tag(MODEL_NAME, best_version, "validated_by", "instructor")

## Step 5 — Load the model by stage AND by alias, and confirm both work

In [ ]:
import mlflow.pyfunc

model_by_stage = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}/Production")
model_by_alias = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")

sample = X_test.iloc[:3]
print("Predictions via stage (Production): ", model_by_stage.predict(sample))
print("Predictions via alias (@champion):  ", model_by_alias.predict(sample))

## Full version history (audit trail, recall Module 1 §3.4)

In [ ]:
# Note: search_model_versions() doesn't always populate `.aliases` reliably,
# so we look aliases up per-version with get_model_version() instead.
for v in client.search_model_versions(f"name='{MODEL_NAME}'"):
    full_version = client.get_model_version(MODEL_NAME, v.version)
    print(f"v{v.version:>2}  stage={v.current_stage:<10}  run_id={v.run_id}  aliases={full_version.aliases}")

print()
print("Registered model aliases (name -> version):", client.get_registered_model(MODEL_NAME).aliases)

---
### ✅ Deliverable checklist
- [ ] `my-classifier` has at least 2 registered versions
- [ ] One version is in `Staging`, one is in `Production`
- [ ] A `@champion` alias is set on the higher-accuracy version
- [ ] Both `models:/my-classifier/Production` and `models:/my-classifier@champion` successfully load and predict
- [ ] Screenshot of the model's version/stage history in the MLflow UI (Models tab)